# Baseline LSTM — Sequential Model

**Author:** Minho  
**Project:** Risk-Adjusted Portfolio Optimization  
**Model:** LSTM (sequential baseline)  
**Dataset:** configurable via `DATASET_NAME` in Config cell (currently `shared_set_1`)
**MLflow run name:** `Minho_Baseline_LSTM`

---

### What this notebook does

This is the simplest working LSTM baseline for the team's shared research framework.
It is intentionally kept minimal and readable — no tricks, no optimizations.

Steps:
1. Load shared dataset via `portfolio_toolkit`
2. Build shared toolkit features + 1 custom feature (`price_accel`)
3. Build per-ticker rolling sequences for the LSTM
4. Train a 2-layer LSTM to predict 5-day forward returns
5. Emit a standardized prediction table
6. Convert predictions → portfolio weights
7. Run the shared backtest
8. Log everything to MLflow as `Minho_Baseline_LSTM`

### How to run

```
cd <repo-root>
source venv312/bin/activate
jupyter notebook MODELS/Minho/baseline_lstm.ipynb
```

Run all cells top to bottom. No hidden state — every cell is self-contained.

## 0. Bootstrap — locate repo root

In [1]:
import os
import sys
from pathlib import Path

# --- If auto-detection fails, set this manually ---
# repo_root = Path('/Users/minhochoi/Portfolio-Optimization-Lib') 
# --------------------------------------------------

def _is_repo_root(p: Path) -> bool:
    return (p / 'pyproject.toml').exists() and (p / 'src' / 'portfolio_toolkit').exists()

if 'repo_root' not in dir() or not _is_repo_root(Path(repo_root)):
    _candidates = [Path.cwd(), *Path.cwd().parents]
    _found = next((p for p in _candidates if _is_repo_root(p)), None)
    if _found is None:
        raise RuntimeError(
            'Cannot find repo root. Uncomment and set repo_root manually at the top of this cell.'
        )
    repo_root = _found

repo_root = Path(repo_root).resolve()
os.chdir(repo_root)

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root:', repo_root)
print('python:   ', sys.executable)

repo_root: /Users/minhochoi/Portfolio-Optimization-Lib
python:    /Users/minhochoi/Portfolio-Optimization-Lib/venv312/bin/python


## 1. Imports

In [2]:
import random
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from portfolio_toolkit import (
    backtest_weights,
    build_features,
    get_dataset_spec,
    custom_dataset,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    slice_split,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_rank_long_only,
    write_backtest_artifacts,
)

warnings.filterwarnings('ignore')
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

/Users/minhochoi/Portfolio-Optimization-Lib/venv312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0 | CUDA: False


--------------------------------------------------------------------------------

### `build_model_features(prices)` & `predict_from_prices(model, prices, dates=None, tickers=None)`

In [3]:
# ── Reusable model functions ───────────────────────────────────────────────────

def build_model_features(prices: pd.DataFrame) -> pd.DataFrame:
    """
    Build the full feature matrix from raw prices.
    Returns a dataframe with columns: date, ticker, *ALL_FEATURES.
    Call this any time you need features — training, validation, or live scoring.
    """
    base_features = [
        'return_1d', 'return_5d', 'return_20d',
        'vol_5d', 'vol_20d',
        'momentum_5d', 'momentum_20d', 'momentum_60d',
        'rsi_14', 'macd_hist', 'bollinger_z_20d',
        'beta_20d_spy', 'excess_return_5d_vs_spy',
        'volume_zscore_20d', 'price_to_sma_20d', 'atr_14',
    ]

    # Shared toolkit features
    feature_frame = build_features(prices, feature_names=base_features)

    # Custom feature: price acceleration (return_5d − return_20d)
    panel = prices.sort_values(['ticker', 'date'])
    ret5  = panel.groupby('ticker')['adj_close'].pct_change(5)
    ret20 = panel.groupby('ticker')['adj_close'].pct_change(20)
    custom_df = panel[['date', 'ticker']].copy()
    custom_df['price_accel'] = (ret5 - ret20).values

    frame = feature_frame.merge(custom_df, on=['date', 'ticker'], how='left')
    return frame


def predict_from_prices(
    model: nn.Module,
    prices: pd.DataFrame,
    train_mean: pd.Series,
    train_std: pd.Series,
    feature_names: list,
    seq_len: int,
    horizon: int,
    dates=None,
    tickers=None,
) -> pd.DataFrame:
    """
    Generate the standard prediction frame from raw prices.

    Parameters
    ----------
    model       : trained BaselineLSTM
    prices      : raw prices dataframe (same format as load_prices output)
    train_mean  : feature means computed on training split (for normalization)
    train_std   : feature stds computed on training split (for normalization)
    feature_names : ordered list of feature column names the model expects
    seq_len     : look-back window length used during training
    horizon     : forecast horizon in trading days
    dates       : optional list/array of dates to filter output (post-inference)
    tickers     : optional list of tickers to filter output (post-inference)

    Returns
    -------
    pd.DataFrame with columns: date, ticker, horizon, expected_return
    """
    # 1. Build features from raw prices
    frame = build_model_features(prices)
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=feature_names)

    # 2. Normalize using the training statistics (no leakage)
    X_scaled = ((frame[feature_names] - train_mean) / train_std).to_numpy(dtype=np.float32)

    # 3. Build per-ticker sequences
    X_seqs, meta = [], []
    frame = frame.reset_index(drop=True)
    for ticker, grp in frame.groupby('ticker', sort=False):
        idx   = grp.index.tolist()
        dates_grp = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            X_seqs.append(X_scaled[rows])
            meta.append((dates_grp[end], ticker))

    if not X_seqs:
        raise ValueError('No sequences could be built — not enough rows per ticker.')

    # 4. Run inference
    model.eval()
    X_tensor = torch.as_tensor(np.stack(X_seqs), dtype=torch.float32)
    with torch.no_grad():
        preds = model(X_tensor.to(next(model.parameters()).device)).cpu().numpy()

    # 5. Assemble prediction frame
    pred_dates, pred_tickers = zip(*meta)
    predictions = pd.DataFrame({
        'date'            : pd.to_datetime(list(pred_dates)),
        'ticker'          : list(pred_tickers),
        'horizon'         : horizon,
        'expected_return' : preds.astype(float),
    })
    predictions = predictions.drop_duplicates(['date', 'ticker', 'horizon'], keep='last')

    # 6. Apply optional filters
    if tickers is not None:
        predictions = predictions[predictions['ticker'].isin(tickers)]
    if dates is not None:
        predictions = predictions[predictions['date'].isin(pd.to_datetime(dates))]

    return predictions.sort_values(['date', 'ticker']).reset_index(drop=True)


print('build_model_features() and predict_from_prices() defined.')

build_model_features() and predict_from_prices() defined.


----------------------------------------------------------------------------------

## 2. Config

All hyperparameters in one place.

In [4]:
'''
DATASET_NAME  = 'shared_set_1'
HORIZON       = 5          # forecast horizon in trading days
RUN_NAME      = 'Minho_Baseline_LSTM'

SEQ_LEN       = 20         # look-back window (trading days)
HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
DROPOUT       = 0.2
LR            = 1e-3
EPOCHS        = 30
BATCH_SIZE    = 512
SEED          = 42

artifact_dir  = repo_root / 'outputs' / RUN_NAME
artifact_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)

# --------------------------------------------------------------------------------
# Exclude benchmark ticker from predictions and portfolio
# SPY is what we're trying to beat — it should not be a holdable position
BENCHMARK_TICKER = spec.benchmark_ticker          # 'SPY'
TRADABLE_TICKERS = [t for t in spec.tickers if t != BENCHMARK_TICKER]

print(f'Benchmark (excluded) : {BENCHMARK_TICKER}')
print(f'Tradable tickers     : {len(TRADABLE_TICKERS)} stocks')
# --------------------------------------------------------------------------------

print(f'Dataset : {DATASET_NAME}  ({len(spec.tickers)} tickers)')
print(f'Train   : {spec.train_start} → {spec.train_end}')
print(f'Val     : {spec.val_start}   → {spec.val_end}')
print(f'Test    : {spec.test_start}  → {spec.test_end}')

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
    else 'mps' if torch.backends.mps.is_available()
)
print(f'Device : {DEVICE}')

'''

"\nDATASET_NAME  = 'shared_set_1'\nHORIZON       = 5          # forecast horizon in trading days\nRUN_NAME      = 'Minho_Baseline_LSTM'\n\nSEQ_LEN       = 20         # look-back window (trading days)\nHIDDEN_SIZE   = 64\nNUM_LAYERS    = 2\nDROPOUT       = 0.2\nLR            = 1e-3\nEPOCHS        = 30\nBATCH_SIZE    = 512\nSEED          = 42\n\nartifact_dir  = repo_root / 'outputs' / RUN_NAME\nartifact_dir.mkdir(parents=True, exist_ok=True)\n\nspec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)\n\n# --------------------------------------------------------------------------------\n# Exclude benchmark ticker from predictions and portfolio\n# SPY is what we're trying to beat — it should not be a holdable position\nBENCHMARK_TICKER = spec.benchmark_ticker          # 'SPY'\nTRADABLE_TICKERS = [t for t in spec.tickers if t != BENCHMARK_TICKER]\n\nprint(f'Benchmark (excluded) : {BENCHMARK_TICKER}')\nprint(f'Tradable tickers     : {len(TRADABLE_TICKERS)} stocks')\n# ---------------------

### Custom subset from `shared_set_1`

In [6]:
from portfolio_toolkit import custom_dataset
from dataclasses import replace
from datetime import date

SUBSET_TICKERS = [
    # Technology
    'AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'AVGO', 'ORCL', 'ADBE', 'CRM', 'AMD', 'INTC',
    # Financials
    'JPM', 'BAC', 'GS', 'MS', 'BLK', 'SPGI', 'V', 'MA',
    # Healthcare
    'LLY', 'UNH', 'JNJ', 'ABBV', 'MRK', 'TMO',
    # Consumer
    'AMZN', 'TSLA', 'HD', 'MCD', 'NKE', 'SBUX',
    # Industrials
    'CAT', 'DE', 'UPS', 'HON', 'GE', 'RTX',
    # Energy
    'XOM', 'CVX', 'COP', 'SLB',
    # Communication
    'DIS', 'NFLX', 'CMCSA', 'T',
    # Utilities / Defensives
    'PG', 'KO', 'PEP', 'WMT', 'COST',
]

dataset = custom_dataset(
    tickers   = SUBSET_TICKERS,
    start     = '2014-01-02',
    end       = '2025-12-31',
    benchmark = 'SPY',
    name      = 'sp500_subset_50',
)

dataset = replace(
    dataset,
    train_start = date(2014, 1,  2),
    train_end   = date(2020, 12, 31),
    val_start   = date(2021, 1,  4),
    val_end     = date(2021, 12, 31),
    test_start  = date(2022, 1,  3),
    test_end    = date(2025, 12, 31),
)

DATASET_NAME     = dataset
HORIZON          = 5
RUN_NAME         = 'Minho_Baseline_LSTM'
SEQ_LEN          = 20
HIDDEN_SIZE      = 64
NUM_LAYERS       = 2
DROPOUT          = 0.2
LR               = 1e-3
EPOCHS           = 30
BATCH_SIZE       = 512
SEED             = 42
REBALANCE_FREQ   = 5
MAX_WEIGHT       = 0.10
TARGET_TYPE      = 'alpha'

artifact_dir     = repo_root / 'outputs' / RUN_NAME
artifact_dir.mkdir(parents=True, exist_ok=True)

spec             = dataset
BENCHMARK_TICKER = spec.benchmark_ticker
TRADABLE_TICKERS = [t for t in spec.tickers if t != BENCHMARK_TICKER]

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)

print(f'Dataset  : {dataset.name}  ({len(TRADABLE_TICKERS)} tradable tickers)')
print(f'Benchmark: {BENCHMARK_TICKER}')
print(f'Train    : {spec.train_start} → {spec.train_end}')
print(f'Val      : {spec.val_start}   → {spec.val_end}')
print(f'Test     : {spec.test_start}  → {spec.test_end}')
print(f'Device   : {DEVICE}')

Dataset  : sp500_subset_50  (50 tradable tickers)
Benchmark: SPY
Train    : 2014-01-02 → 2020-12-31
Val      : 2021-01-04   → 2021-12-31
Test     : 2022-01-03  → 2025-12-31
Device   : mps


## 3. Load Prices

In [7]:
prices = load_prices(DATASET_NAME, repo_root=repo_root)
print('Shape     :', prices.shape)
print('Date range:', prices['date'].min().date(), '→', prices['date'].max().date())
prices.head(3)

Shape     : (150900, 8)
Date range: 2014-01-02 → 2025-12-31


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140661,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764154,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855570,412610800


## 4. Features — Shared Toolkit + 1 Custom

**Custom feature: `price_accel`**  
Defined as `return_5d − return_20d`. Captures whether short-term momentum is
accelerating (positive) or decelerating (negative) relative to the medium-term trend.
Particularly informative for an LSTM because the model can learn how acceleration
evolves across the look-back window.

In [8]:
# Now just one line instead of 15 lines of feature engineering
frame = build_model_features(prices)
ALL_FEATURES = [
    'return_1d', 'return_5d', 'return_20d',
    'vol_5d', 'vol_20d',
    'momentum_5d', 'momentum_20d', 'momentum_60d',
    'rsi_14', 'macd_hist', 'bollinger_z_20d',
    'beta_20d_spy', 'excess_return_5d_vs_spy',
    'volume_zscore_20d', 'price_to_sma_20d', 'atr_14',
    'price_accel',  # custom feature
]

print(f'Features: {len(ALL_FEATURES)}  (16 shared + 1 custom: price_accel)')
frame[['date', 'ticker', 'price_accel']].dropna().head(3)

Features: 17  (16 shared + 1 custom: price_accel)


,date,ticker,price_accel
20,2014-01-31,AAPL,0.011701
21,2014-02-03,AAPL,-0.016032
22,2014-02-04,AAPL,0.069125


## 5. Targets + Train / Val / Test Split

In [9]:
TARGET_COL = f'forward_alpha_{HORIZON}d_vs_spy'
targets    = make_forward_alpha_target(prices, horizon=HORIZON)

target_frame = frame.merge(targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

train = slice_split(target_frame, DATASET_NAME, 'train', repo_root=repo_root)
val   = slice_split(target_frame, DATASET_NAME, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, DATASET_NAME, 'test',  repo_root=repo_root)

# Normalize using train statistics only (no leakage)
train_mean = train[ALL_FEATURES].mean()
train_std  = train[ALL_FEATURES].std(ddof=0).replace(0.0, 1.0)

def scale(df):
    return ((df[ALL_FEATURES] - train_mean) / train_std).to_numpy(dtype=np.float32)

X_train, X_val, X_test = scale(train), scale(val), scale(test)
y_train = train[TARGET_COL].to_numpy(dtype=np.float32)
y_val   = val[TARGET_COL].to_numpy(dtype=np.float32)

print(f'Train : {len(train):>6,} rows    X_train : {X_train.shape}')
print(f'Val   : {len(val):>6,} rows    X_val   : {X_val.shape}')
print(f'Test  : {len(test):>6,} rows    X_test  : {X_test.shape}')

Train : 85,148 rows    X_train : (85148, 17)
Val   : 12,600 rows    X_val   : (12600, 17)
Test  : 49,892 rows    X_test  : (49892, 17)


## 6. Build Per-Ticker Sequences

LSTMs consume `(batch, seq_len, n_features)` tensors. We build overlapping
windows of length `SEQ_LEN` **within each ticker** so sequences never span
ticker boundaries. The label is the target at the last timestep of each window.

In [10]:
def make_sequences(
    df: pd.DataFrame,
    X: np.ndarray,
    y: np.ndarray,
    seq_len: int,
):
    """Return (X_seq, y_seq, meta) where meta is list of (date, ticker)."""
    Xs, ys, meta = [], [], []
    df = df.reset_index(drop=True)
    for ticker, grp in df.groupby('ticker', sort=False):
        idx   = grp.index.tolist()
        dates = grp['date'].tolist()
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx)):
            rows = idx[end - seq_len + 1 : end + 1]
            Xs.append(X[rows])
            ys.append(y[idx[end]])
            meta.append((dates[end], ticker))
    return (
        np.stack(Xs).astype(np.float32),
        np.array(ys, dtype=np.float32),
        meta,
    )

print('Building sequences ...')
X_tr_seq, y_tr_seq, _          = make_sequences(train, X_train, y_train, SEQ_LEN)
X_va_seq, y_va_seq, _          = make_sequences(val,   X_val,   y_val,   SEQ_LEN)
X_te_seq, y_te_seq, meta_test  = make_sequences(test,  X_test,  test[TARGET_COL].to_numpy(np.float32), SEQ_LEN)

print(f'Train sequences : {X_tr_seq.shape}')   # (N, SEQ_LEN, F)
print(f'Val sequences   : {X_va_seq.shape}')
print(f'Test sequences  : {X_te_seq.shape}')

Building sequences ...
Train sequences : (84198, 20, 17)
Val sequences   : (11650, 20, 17)
Test sequences  : (48942, 20, 17)


## 7. LSTM Model

A minimal 2-layer stacked LSTM. Input → LSTM → last hidden state → linear head → scalar return prediction.
Causal (not bidirectional). Gradient clipping applied during training.

In [11]:
class BaselineLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)       # h_n: (num_layers, batch, hidden)
        return self.head(h_n[-1]).squeeze(-1)  # scalar per sequence


def _eval_loss_batched(model, X, y, device, batch_size=2048):
    """Compute MSE loss in batches — avoids loading all sequences onto MPS at once."""
    loss_fn = nn.MSELoss(reduction='sum')
    total, n = 0.0, 0
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            Xb = torch.as_tensor(X[i:i+batch_size], dtype=torch.float32, device=device)
            yb = torch.as_tensor(y[i:i+batch_size], dtype=torch.float32, device=device)
            total += float(loss_fn(model(Xb), yb))
            n     += len(Xb)
    return total / n


def train_lstm(model, X_tr, y_tr, X_va, y_va, epochs, batch_size, lr, device):
    """Train and return loss history DataFrame."""
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.MSELoss()
    loader    = DataLoader(
        TensorDataset(
            torch.as_tensor(X_tr, dtype=torch.float32),
            torch.as_tensor(y_tr, dtype=torch.float32),
        ),
        batch_size=batch_size, shuffle=True,
    )
    history = []
    for epoch in range(epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Batched eval — never loads full dataset onto MPS at once
        tr_loss = _eval_loss_batched(model, X_tr, y_tr, device)
        va_loss = _eval_loss_batched(model, X_va, y_va, device)

        history.append({'epoch': epoch + 1, 'train_loss': tr_loss, 'val_loss': va_loss})
        if (epoch + 1) % 5 == 0:
            print(f'  epoch {epoch+1:3d}/{epochs}  train={tr_loss:.6f}  val={va_loss:.6f}')

    return pd.DataFrame(history)


print('Device:', DEVICE)
print(f'Input size: {X_tr_seq.shape[2]}')

Device: mps
Input size: 17


## 8. Train

In [12]:
model = BaselineLSTM(
    input_size  = X_tr_seq.shape[2],
    hidden_size = HIDDEN_SIZE,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
)

print(f'Training on {X_tr_seq.shape[0]:,} sequences for {EPOCHS} epochs ...')
history = train_lstm(
    model, X_tr_seq, y_tr_seq, X_va_seq, y_va_seq,
    epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
)

best_val = history['val_loss'].min()
print(f'\nBest val MSE: {best_val:.8f}')
history.tail()

Training on 84,198 sequences for 30 epochs ...
  epoch   5/30  train=0.001719  val=0.001747
  epoch  10/30  train=0.001087  val=0.001188
  epoch  15/30  train=0.001052  val=0.001173
  epoch  20/30  train=0.001017  val=0.001144
  epoch  25/30  train=0.001043  val=0.001250
  epoch  30/30  train=0.000980  val=0.001258

Best val MSE: 0.00114120


,epoch,train_loss,val_loss
25,26,0.001173,0.001372
26,27,0.001035,0.001253
27,28,0.001047,0.001290
28,29,0.001069,0.001317
29,30,0.000980,0.001258


In [13]:
# ── Revision: Validation Diagnostics ────────────────────────────────────────
from scipy import stats

print('=' * 60)
print('VALIDATION DIAGNOSTICS')
print('=' * 60)

# ── 1. Naive baseline comparison ──────────────────────────────────────────────
# If your LSTM can't beat "always predict zero" or "always predict
# the historical mean", it has no useful signal at all.

print('\n── 1. Naive Baseline Comparison ──')

naive_zero_mse  = float(np.mean(y_va_seq ** 2))
naive_mean      = float(y_tr_seq.mean())
naive_mean_mse  = float(np.mean((y_va_seq - naive_mean) ** 2))
lstm_val_mse    = float(best_val)

print(f'  Naive zero-predict MSE  : {naive_zero_mse:.8f}')
print(f'  Naive mean-predict MSE  : {naive_mean_mse:.8f}')
print(f'  LSTM val MSE (best)     : {lstm_val_mse:.8f}')
print()

if lstm_val_mse < naive_zero_mse:
    improvement = (1 - lstm_val_mse / naive_zero_mse) * 100
    print(f'  ✓ LSTM beats zero-predict by {improvement:.1f}%')
else:
    print(f'  ✗ LSTM does NOT beat zero-predict — no useful signal detected')

if lstm_val_mse < naive_mean_mse:
    improvement = (1 - lstm_val_mse / naive_mean_mse) * 100
    print(f'  ✓ LSTM beats mean-predict by {improvement:.1f}%')
else:
    print(f'  ✗ LSTM does NOT beat mean-predict — no useful signal detected')

# ── 2. Directional accuracy ────────────────────────────────────────────────────
# What % of the time does the model correctly predict up vs down?
# 50% = random. Anything above ~52% on financial data is meaningful.

print('\n── 2. Directional Accuracy ──')

# Get val set predictions from the trained model
# Replaced from:
# val_preds = model(
#     torch.as_tensor(X_va_seq, dtype=torch.float32, device=DEVICE)
# ).cpu().numpy()

# With this batched version:
model.eval()
val_preds_list = []
with torch.no_grad():
    for i in range(0, len(X_va_seq), 2048):
        Xb = torch.as_tensor(X_va_seq[i:i+2048], dtype=torch.float32, device=DEVICE)
        val_preds_list.append(model(Xb).cpu().numpy())
val_preds = np.concatenate(val_preds_list)

actual_direction    = np.sign(y_va_seq)
predicted_direction = np.sign(val_preds)

# Exclude flat actual returns (sign = 0) from directional accuracy
nonzero_mask   = actual_direction != 0
dir_accuracy   = float(
    (actual_direction[nonzero_mask] == predicted_direction[nonzero_mask]).mean()
)

print(f'  Directional accuracy : {dir_accuracy:.4f}  ({dir_accuracy*100:.1f}%)')
if dir_accuracy > 0.52:
    print(f'  ✓ Above 52% threshold — model shows directional signal')
elif dir_accuracy > 0.50:
    print(f'  ~ Marginally above random — weak signal')
else:
    print(f'  ✗ At or below random (50%) — no directional signal')

# ── 3. Information Coefficient (IC) ───────────────────────────────────────────
# Rank correlation between predicted and actual returns.
# IC > 0.05 is considered meaningful in practice.
# IC > 0.10 is strong for a single-factor model.

print('\n── 3. Information Coefficient (IC) ──')

ic, ic_pvalue = stats.spearmanr(val_preds, y_va_seq)
ic = float(ic)

print(f'  IC (Spearman rank corr) : {ic:.4f}')
print(f'  IC p-value              : {ic_pvalue:.4f}')

if ic > 0.10:
    print(f'  ✓ Strong IC — good rank-ordering signal')
elif ic > 0.05:
    print(f'  ✓ Meaningful IC — model has useful ranking ability')
elif ic > 0.02:
    print(f'  ~ Weak IC — marginal signal, may not be reliable')
else:
    print(f'  ✗ IC near zero — model cannot rank stocks meaningfully')

# ── 4. IC by month (stability check) ──────────────────────────────────────────
# A model that works sometimes but not others is not reliable.
# We want IC to be consistently positive across months, not just on average.

print('\n── 4. IC Stability by Month ──')

# Rebuild val metadata to get dates per sequence
_, _, meta_val = make_sequences(val, X_val, y_val, SEQ_LEN)
val_dates = pd.to_datetime([m[0] for m in meta_val])

ic_by_month = (
    pd.DataFrame({
        'date'     : val_dates,
        'predicted': val_preds,
        'actual'   : y_va_seq,
    })
    .assign(month=lambda df: df['date'].dt.to_period('M'))
    .groupby('month')
    .apply(lambda g: float(stats.spearmanr(g['predicted'], g['actual'])[0]))
    .rename('monthly_ic')
)

positive_months = (ic_by_month > 0).sum()
total_months    = len(ic_by_month)
ic_hit_rate     = positive_months / total_months

print(f'  Mean monthly IC   : {ic_by_month.mean():.4f}')
print(f'  Std monthly IC    : {ic_by_month.std():.4f}')
print(f'  IC hit rate       : {positive_months}/{total_months} months positive  ({ic_hit_rate:.1%})')
print()
print(ic_by_month.to_string())

if ic_hit_rate >= 0.60:
    print(f'\n  ✓ IC positive in {ic_hit_rate:.0%} of months — reasonably stable')
else:
    print(f'\n  ✗ IC positive in only {ic_hit_rate:.0%} of months — unstable signal')

# ── 5. Performance by date regime ─────────────────────────────────────────────
# Does the model work in all market conditions or only in specific regimes?
# We split val period into high-vol and low-vol regimes using SPY returns.

print('\n── 5. IC by Volatility Regime ──')

spy_val = (
    prices[prices['ticker'] == 'SPY']
    .set_index('date')['adj_close']
    .pct_change()
    .rolling(20)
    .std()
    .rename('spy_vol_20d')
)

regime_df = pd.DataFrame({
    'date'     : val_dates,
    'predicted': val_preds,
    'actual'   : y_va_seq,
}).merge(
    spy_val.reset_index().rename(columns={'date': 'date'}),
    on='date', how='left'
)

vol_median = regime_df['spy_vol_20d'].median()
regime_df['regime'] = np.where(
    regime_df['spy_vol_20d'] >= vol_median, 'high_vol', 'low_vol'
)

for regime, grp in regime_df.groupby('regime'):
    if len(grp) < 10:
        continue
    regime_ic, _ = stats.spearmanr(grp['predicted'], grp['actual'])
    print(f'  {regime:<10s}: IC = {float(regime_ic):.4f}  (n={len(grp):,})')

# ── Summary table ──────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('DIAGNOSTIC SUMMARY')
print('=' * 60)
print(f'  Beats zero-predict    : {"Yes" if lstm_val_mse < naive_zero_mse else "No"}')
print(f'  Beats mean-predict    : {"Yes" if lstm_val_mse < naive_mean_mse else "No"}')
print(f'  Directional accuracy  : {dir_accuracy*100:.1f}%')
print(f'  IC (overall)          : {ic:.4f}')
print(f'  IC hit rate (monthly) : {ic_hit_rate:.1%}')
print('=' * 60)

VALIDATION DIAGNOSTICS

── 1. Naive Baseline Comparison ──
  Naive zero-predict MSE  : 0.00108578
  Naive mean-predict MSE  : 0.00108509
  LSTM val MSE (best)     : 0.00114120

  ✗ LSTM does NOT beat zero-predict — no useful signal detected
  ✗ LSTM does NOT beat mean-predict — no useful signal detected

── 2. Directional Accuracy ──
  Directional accuracy : 0.4898  (49.0%)
  ✗ At or below random (50%) — no directional signal

── 3. Information Coefficient (IC) ──
  IC (Spearman rank corr) : -0.0282
  IC p-value              : 0.0023
  ✗ IC near zero — model cannot rank stocks meaningfully

── 4. IC Stability by Month ──
  Mean monthly IC   : -0.0246
  Std monthly IC    : 0.0952
  IC hit rate       : 5/11 months positive  (45.5%)

month
2021-02    0.084223
2021-03   -0.109622
2021-04   -0.038982
2021-05   -0.050931
2021-06   -0.151474
2021-07    0.098339
2021-08    0.084641
2021-09   -0.167294
2021-10   -0.065956
2021-11    0.043559
2021-12    0.002896
Freq: M

  ✗ IC positive in only 

---------------------------------------------------------------------------------

### Checkpoint saving
Save Model Weights + `log_model_submission`

In [14]:
# ── Save model checkpoint + metadata ──────────────────────────────────────────
import json
import torch

checkpoint_dir = artifact_dir / 'model'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Single bundled checkpoint — contains everything needed to reload the model
checkpoint_path = checkpoint_dir / 'lstm_checkpoint.pt'
torch.save(
    {
        'state_dict'          : model.state_dict(),
        'feature_names'       : ALL_FEATURES,
        'horizon'             : HORIZON,
        'rebalance_frequency' : 'every_5_trading_days',
        'model_config': {
            'architecture' : 'BaselineLSTM',
            'input_dim'    : len(ALL_FEATURES),
            'hidden_dim'   : HIDDEN_SIZE,
            'num_layers'   : NUM_LAYERS,
            'dropout'      : DROPOUT,
        },
        'preprocessing': {
            'scaler' : 'train_mean_std',
            'means'  : train_mean.to_dict(),
            'stds'   : train_std.to_dict(),
        },
        'training': {
            'epochs'       : EPOCHS,
            'batch_size'   : BATCH_SIZE,
            'lr'           : LR,
            'seed'         : SEED,
            'best_val_mse' : round(float(best_val), 8),
        },
        'data': {
            'dataset_name'   : dataset.name if hasattr(dataset, 'name') else str(dataset),
            'tickers'        : list(spec.tickers),
            'benchmark'      : str(spec.benchmark_ticker),
            'train_end'      : str(spec.train_end),
            'val_end'        : str(spec.val_end),
            'test_start'     : str(spec.test_start),
            'target'         : TARGET_COL,
            'seq_len'        : SEQ_LEN,
            'custom_features': ['price_accel'],
        },
        'source_notebook' : 'MODELS/Minho/baseline_lstm.ipynb',
        'run_name'        : RUN_NAME,
        'author'          : 'Minho',
    },
    checkpoint_path,
)
print(f'Bundled checkpoint saved → {checkpoint_path}')

# Reconstruction recipe
recipe = """
# How to reproduce Minho baseline LSTM predictions from scratch
#
# Requirements: repo installed, venv312 active, MLflow artifacts downloaded
#
# Steps:
#   1. Load the bundled checkpoint
#      bundle = torch.load('lstm_checkpoint.pt', map_location='cpu')
#
#   2. Rebuild model architecture
#      model = BaselineLSTM(
#          input_size  = bundle['model_config']['input_dim'],
#          hidden_size = bundle['model_config']['hidden_dim'],
#          num_layers  = bundle['model_config']['num_layers'],
#          dropout     = bundle['model_config']['dropout'],
#      )
#      model.load_state_dict(bundle['state_dict'])
#      model.eval()
#
#   3. Load preprocessing
#      train_mean = pd.Series(bundle['preprocessing']['means'])
#      train_std  = pd.Series(bundle['preprocessing']['stds'])
#
#   4. Load prices and call predict_from_prices()
#      prices = load_prices(bundle['data']['dataset_name'])
#      predictions = predict_from_prices(
#          model=model, prices=prices,
#          train_mean=train_mean, train_std=train_std,
#          feature_names=bundle['feature_names'],
#          seq_len=bundle['data']['seq_len'],
#          horizon=bundle['horizon'],
#      )
"""
recipe_path = checkpoint_dir / 'REPRODUCE.py'
recipe_path.write_text(recipe)
print(f'Reproduction recipe saved → {recipe_path}')

print('\nAll model artifacts saved:')
for p in sorted(checkpoint_dir.iterdir()):
    print(f'  {p.name}')

Bundled checkpoint saved → /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/model/lstm_checkpoint.pt
Reproduction recipe saved → /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/model/REPRODUCE.py

All model artifacts saved:
  REPRODUCE.py
  lstm_checkpoint.pt
  lstm_weights.pt
  model_metadata.json
  norm_stats.json


---------------------------------------------------------------------------------

## 9. Predictions

In [15]:
# Generate predictions — SPY explicitly excluded via TRADABLE_TICKERS
predictions = predict_from_prices(
    model         = model,
    prices        = prices,
    train_mean    = train_mean,
    train_std     = train_std,
    feature_names = ALL_FEATURES,
    seq_len       = SEQ_LEN,
    horizon       = HORIZON,
    tickers       = TRADABLE_TICKERS,
)

# Filter to test split dates only
test_dates = set(test['date'].unique())
predictions = predictions[
    predictions['date'].isin(test_dates)
].reset_index(drop=True)

# Filter to rebalance dates only — must match portfolio weight dates
# so MLflow predictions and weights are consistent
all_pred_dates = sorted(predictions['date'].unique())
rebal_dates    = set(pd.DatetimeIndex(all_pred_dates)[::REBALANCE_FREQ])
predictions    = predictions[
    predictions['date'].isin(rebal_dates)
].reset_index(drop=True)

# Validate
assert BENCHMARK_TICKER not in predictions['ticker'].values, \
    f'{BENCHMARK_TICKER} found in predictions — exclusion failed'

print('Predictions shape :', predictions.shape)
print('Rebalance dates   :', len(rebal_dates))
print('SPY in predictions:', BENCHMARK_TICKER in predictions['ticker'].values)
predictions.head()

Predictions shape : (9799, 4)
Rebalance dates   : 200
SPY in predictions: False


,date,ticker,horizon,expected_return
0,2022-01-03,AAPL,5,0.005763
1,2022-01-03,ABBV,5,0.016938
2,2022-01-03,ADBE,5,0.008293
3,2022-01-03,AMD,5,-0.017889
4,2022-01-03,AMZN,5,0.006813


## 10. Portfolio Weights

In [16]:
# ── Portfolio Weights ─────────────────────────────────────────────────

portfolio = weights_from_predictions_rank_long_only(
    predictions,
    score_column  = 'expected_return',
    dataset_name  = DATASET_NAME,
    strategy_name = RUN_NAME,
)

# Apply max weight cap — prevents any single stock dominating
raw_w = portfolio.weights.copy()
raw_w = raw_w.clip(upper=MAX_WEIGHT)
row_sums = raw_w.sum(axis=1).replace(0.0, 1.0)
raw_w = raw_w.div(row_sums, axis=0)

# Apply weekly rebalancing — only keep weight rows every REBALANCE_FREQ days
# This reduces turnover and sensitivity to daily noise
all_dates    = raw_w.index.sort_values()
rebal_dates  = all_dates[::REBALANCE_FREQ]
raw_w        = raw_w.loc[rebal_dates]

# Validate
assert np.allclose(raw_w.sum(axis=1).to_numpy(float), 1.0, atol=1e-6), \
    'Weight rows do not sum to 1.0 after capping'
assert (raw_w <= MAX_WEIGHT + 1e-6).all().all(), \
    'Max weight cap not applied correctly'

# Rebuild portfolio object
from portfolio_toolkit.contracts import PortfolioWeights as _PW
clean_portfolio = _PW(
    weights       = raw_w,
    dataset_name  = DATASET_NAME.name if hasattr(DATASET_NAME, 'name') else DATASET_NAME,
    strategy_name = RUN_NAME,
    metadata      = {'rebalance_freq': REBALANCE_FREQ, 'max_weight': MAX_WEIGHT},
)

validated_weights = validate_weights_frame(clean_portfolio.weights)
print('Weights shape  :', validated_weights.shape)
print('Max weight     :', validated_weights.max().max())
print('Rebal dates    :', len(rebal_dates))
print('Row sums OK    :', np.allclose(validated_weights.sum(axis=1), 1.0))

Weights shape  : (40, 49)
Max weight     : 0.04000000000000001
Rebal dates    : 40
Row sums OK    : True


----------------------------------------------------------------------------------------

## 11. Backtest

In [17]:
# ── Backtest ──────────────────────────────────────────────────────────
import bt
from portfolio_toolkit.backtest import (
    _pivot_prices,
    _align_weights_to_prices,
    _mask_unavailable_weights,
    _make_bt_strategy,
    _commission_fn,
    _compute_turnover,
)
from portfolio_toolkit.reporting import build_metrics
from portfolio_toolkit.contracts import BacktestResult

# ── Step 1: Strict price cleaning ─────────────────────────────────────────────
price_wide_raw    = _pivot_prices(prices)
price_wide_filled = price_wide_raw.ffill(limit=5).bfill(limit=5)

clean_mask  = price_wide_filled.notna().all(axis=0)
price_wide  = price_wide_filled.loc[:, clean_mask]

dropped = sorted(set(price_wide_raw.columns) - set(price_wide.columns))
print(f'Tickers dropped (missing prices) : {len(dropped)}')
if dropped:
    print(f'  {dropped}')

available_tickers = [t for t in TRADABLE_TICKERS if t in price_wide.columns]
print(f'Clean tickers for backtest       : {len(available_tickers)}')

assert price_wide[available_tickers].isna().sum().sum() == 0, \
    'NaNs still present — bt will crash'
print('Price matrix is clean — no NaNs detected.')

# ── Step 2: Align and mask strategy weights ────────────────────────────────────
raw_weights     = validate_weights_frame(clean_portfolio.weights)
aligned_weights = _align_weights_to_prices(
    raw_weights.loc[:, [t for t in available_tickers if t in raw_weights.columns]],
    price_wide.index,
)
aligned_weights = _mask_unavailable_weights(
    aligned_weights, price_wide.loc[:, available_tickers]
)

# ── Step 3: Equal weight baseline (no SPY) ────────────────────────────────────
eq_value      = 1.0 / len(available_tickers)
equal_weights = pd.DataFrame(
    eq_value,
    index   = aligned_weights.index,
    columns = available_tickers,
    dtype   = float,
)
equal_weights.index.name = 'date'
equal_weights = _align_weights_to_prices(equal_weights, price_wide.index)
equal_weights = _mask_unavailable_weights(
    equal_weights, price_wide.loc[:, available_tickers]
)

# ── Step 4: SPY benchmark ──────────────────────────────────────────────────────
spy_bench = pd.DataFrame(
    {'SPY': 1.0},
    index = aligned_weights.index,
)
spy_bench.index.name = 'date'
spy_bench = _align_weights_to_prices(spy_bench, price_wide.index)

cost_fn = _commission_fn(spec.cost_bps)

# ── Step 5: Run bt backtests ───────────────────────────────────────────────────
print('\nRunning backtests ...')
backtests = [
    bt.Backtest(
        _make_bt_strategy(RUN_NAME, aligned_weights),
        price_wide[available_tickers],
        commissions=cost_fn,
        integer_positions=False,
    ),
    bt.Backtest(
        _make_bt_strategy('SPY', spy_bench),
        price_wide[['SPY']],
        commissions=cost_fn,
        integer_positions=False,
    ),
    bt.Backtest(
        _make_bt_strategy('equal_weight', equal_weights),
        price_wide[available_tickers],
        commissions=cost_fn,
        integer_positions=False,
    ),
]

bt_result = bt.run(*backtests)

# ── Step 6: Assemble BacktestResult ───────────────────────────────────────────
nav      = bt_result.prices[RUN_NAME].rename('nav')
returns  = nav.pct_change().fillna(0.0).rename('returns')
turnover = _compute_turnover(aligned_weights)

benchmark_returns = pd.DataFrame(index=nav.index)
for col in bt_result.prices.columns:
    if col == RUN_NAME:
        continue
    benchmark_returns[col] = bt_result.prices[col].pct_change().fillna(0.0)

result = BacktestResult(
    strategy_name    = RUN_NAME,
    dataset_name     = dataset.name if hasattr(dataset, 'name') else str(dataset),
    weights          = aligned_weights,
    nav              = nav,
    returns          = returns,
    turnover         = turnover,
    benchmark_returns= benchmark_returns,
    metrics          = {},
)
result.metrics = build_metrics(result)

print('Backtest complete.')
print()
for k, v in sorted(result.metrics.items()):
    print(f'  {k:<35s}: {v:.6f}')

Tickers dropped (missing prices) : 0
Clean tickers for backtest       : 49
Price matrix is clean — no NaNs detected.

Running backtests ...
Backtest complete.

  annual_excess_return_vs_benchmark  : 0.076444
  annual_return                      : 0.184755
  annual_volatility                  : 0.179582
  average_turnover                   : 0.325510
  benchmark_annual_return            : 0.108312
  benchmark_annual_volatility        : 0.179691
  benchmark_max_drawdown             : -0.244964
  benchmark_sharpe                   : 0.602767
  benchmark_total_return             : 0.507582
  calmar                             : 0.885351
  evaluation_trading_days            : 1003.000000
  evaluation_years                   : 3.991786
  excess_return_vs_benchmark         : 0.459897
  max_drawdown                       : -0.208680
  sharpe                             : 1.028810
  sharpe_vs_benchmark                : 0.426043
  sortino                            : 1.424992
  total_return     

------------------------------------------------------------------------------------------

In [19]:
# ── Benchmark Comparison Table ─────────────────────────────────────────────────
from math import sqrt

print('=' * 70)
print('BENCHMARK COMPARISON')
print('=' * 70)

# ── 1. Extract benchmark metrics from Cell 11 result ──────────────────────────
spy_returns = result.benchmark_returns['SPY'] \
    if 'SPY' in result.benchmark_returns.columns \
    else result.benchmark_returns.iloc[:, 0]

eq_returns = result.benchmark_returns['equal_weight'] \
    if 'equal_weight' in result.benchmark_returns.columns \
    else None

spy_nav = (1.0 + spy_returns).cumprod()
spy_nav = spy_nav / spy_nav.iloc[0]

if eq_returns is not None:
    eq_nav = (1.0 + eq_returns).cumprod()
    eq_nav = eq_nav / eq_nav.iloc[0]
else:
    eq_returns = spy_returns
    eq_nav     = spy_nav
    print('Warning: equal_weight not found in benchmark_returns, using SPY as proxy')

# ── 2. Metrics function ────────────────────────────────────────────────────────
def compute_metrics(nav: pd.Series, returns: pd.Series) -> dict:
    total_return  = float(nav.iloc[-1] / nav.iloc[0] - 1.0)
    n_days        = max(len(returns), 1)
    annual_return = float((1.0 + total_return) ** (252.0 / n_days) - 1.0)
    annual_vol    = float(returns.std(ddof=0) * sqrt(252))
    sharpe        = annual_return / annual_vol if annual_vol > 0 else 0.0
    downside      = returns[returns < 0]
    sortino_vol   = float(downside.std(ddof=0) * sqrt(252)) if len(downside) else 0.0
    sortino       = annual_return / sortino_vol if sortino_vol > 0 else 0.0
    drawdown      = nav / nav.cummax() - 1.0
    max_dd        = float(drawdown.min())
    calmar        = annual_return / abs(max_dd) if max_dd < 0 else 0.0
    return {
        'Total Return' : total_return,
        'Annual Return': annual_return,
        'Annual Vol'   : annual_vol,
        'Sharpe'       : sharpe,
        'Sortino'      : sortino,
        'Max Drawdown' : max_dd,
        'Calmar'       : calmar,
    }

# ── 3. Compute metrics ─────────────────────────────────────────────────────────
lstm_metrics = compute_metrics(result.nav / result.nav.iloc[0], result.returns)
spy_metrics  = compute_metrics(spy_nav, spy_returns)
eq_metrics   = compute_metrics(eq_nav, eq_returns)

# ── 4. Excess metrics ──────────────────────────────────────────────────────────
def excess(a, b): return a - b

lstm_vs_spy = {k: excess(lstm_metrics[k], spy_metrics[k]) for k in lstm_metrics}
lstm_vs_eq  = {k: excess(lstm_metrics[k], eq_metrics[k])  for k in lstm_metrics}

# ── 5. Print comparison table ──────────────────────────────────────────────────
pct_metrics = {'Total Return', 'Annual Return', 'Annual Vol', 'Max Drawdown'}
strategies  = ['LSTM (Minho)', 'SPY Buy & Hold', 'Equal Weight']
metrics_all = [lstm_metrics, spy_metrics, eq_metrics]

header = f'{"Metric":<22s}' + ''.join(f'{s:>18s}' for s in strategies)
print(f'\n{header}')
print('-' * (22 + 18 * len(strategies)))

for metric in lstm_metrics:
    row = f'{metric:<22s}'
    for m in metrics_all:
        val = m[metric]
        row += f'{val:>17.1%} ' if metric in pct_metrics else f'{val:>17.4f} '
    print(row)

# ── 6. Excess vs benchmarks ────────────────────────────────────────────────────
for label, diff in [('Excess vs SPY', lstm_vs_spy), ('Excess vs Equal Weight', lstm_vs_eq)]:
    print(f'\n{label}')
    print('-' * 40)
    for metric in ['Total Return', 'Annual Return', 'Sharpe', 'Max Drawdown']:
        val  = diff[metric]
        fmt  = f'{val:>+.1%}' if metric in pct_metrics else f'{val:>+.4f}'
        flag = '✓' if val > 0 else '✗'
        print(f'  {flag} {metric:<20s}: {fmt}')

# ── 7. Turnover ────────────────────────────────────────────────────────────────
print()
print('── Turnover ──')
print(f'  LSTM avg daily turnover  : {result.turnover.mean():.4f}')
print(f'  SPY turnover             : ~0.0000  (buy and hold)')

# ── 8. Cache for MLflow ────────────────────────────────────────────────────────
_comparison_metrics_cache = {
    'lstm_annual_return'          : lstm_metrics['Annual Return'],
    'lstm_sharpe'                 : lstm_metrics['Sharpe'],
    'lstm_max_drawdown'           : lstm_metrics['Max Drawdown'],
    'spy_annual_return'           : spy_metrics['Annual Return'],
    'spy_sharpe'                  : spy_metrics['Sharpe'],
    'eq_annual_return'            : eq_metrics['Annual Return'],
    'eq_sharpe'                   : eq_metrics['Sharpe'],
    'excess_annual_return_vs_spy' : lstm_vs_spy['Annual Return'],
    'excess_sharpe_vs_spy'        : lstm_vs_spy['Sharpe'],
    'excess_annual_return_vs_eq'  : lstm_vs_eq['Annual Return'],
    'excess_sharpe_vs_eq'         : lstm_vs_eq['Sharpe'],
}

print()
print('=' * 70)
print('BENCHMARK COMPARISON COMPLETE')
print('=' * 70)

BENCHMARK COMPARISON

Metric                      LSTM (Minho)    SPY Buy & Hold      Equal Weight
----------------------------------------------------------------------------
Total Return                      96.6%             50.8%             72.8% 
Annual Return                      5.8%              3.5%              4.7% 
Annual Vol                        10.4%             10.4%              9.8% 
Sharpe                           0.5590            0.3359            0.4776 
Sortino                          0.4469            0.2673            0.3749 
Max Drawdown                     -20.9%            -24.6%            -21.9% 
Calmar                           0.2781            0.1419            0.2135 

Excess vs SPY
----------------------------------------
  ✓ Total Return        : +45.8%
  ✓ Annual Return       : +2.3%
  ✓ Sharpe              : +0.2231
  ✓ Max Drawdown        : +3.7%

Excess vs Equal Weight
----------------------------------------
  ✓ Total Return        : +23.8%


------------------------------------------------------------------------------------------

## 12. Write Artifacts (QuantStats report + parquet files)

In [20]:
artifact_paths = write_backtest_artifacts(result, artifact_dir)

for key, path in artifact_paths.items():
    exists = '✓' if Path(path).exists() else '✗'
    print(f'  {exists} {key:<20s}: {path}')

  ✓ weights             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/weights.parquet
  ✓ nav                 : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/nav.parquet
  ✓ returns             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/returns.parquet
  ✓ turnover            : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/turnover.parquet
  ✓ benchmarks          : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/benchmarks.parquet
  ✓ metrics             : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/metrics.json
  ✓ metrics_table       : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/metrics_table.parquet
  ✓ quantstats_report   : /Users/minhochoi/Portfolio-Optimization-Lib/outputs/Minho_Baseline_LSTM/quantstats.html


## 13. Log to MLflow as `Minho_Baseline_LSTM`

In [30]:
import mlflow
from portfolio_toolkit.tracking import log_model_submission

mlflow_layout = init_mlflow(repo_root)
print('Tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name     = RUN_NAME,
    dataset_name = dataset,        # ← pass the object, not dataset.name
    tags={
        'author'        : 'Minho',
        'model_family'  : 'sequential_lstm',
        'workflow'      : 'baseline_lstm',
        'horizon'       : str(HORIZON),
        'dataset'       : dataset.name,
        'project'       : 'risk_adjusted_portfolio_optimization',
    },
    repo_root=repo_root,
):
    # Params
    mlflow.log_params({
        'run_name'            : RUN_NAME,
        'dataset'             : dataset.name if hasattr(dataset, 'name') else str(DATASET_NAME),
        'horizon'             : HORIZON,
        'seq_len'             : SEQ_LEN,
        'hidden_size'         : HIDDEN_SIZE,
        'num_layers'          : NUM_LAYERS,
        'dropout'             : DROPOUT,
        'lr'                  : LR,
        'epochs'              : EPOCHS,
        'batch_size'          : BATCH_SIZE,
        'n_features'          : len(ALL_FEATURES),
        'custom_features'     : 'price_accel',
        'portfolio_builder'   : 'rank_long_only',
        'best_val_mse'        : round(float(best_val), 8),
        'target_type'         : TARGET_TYPE,
        'rebalance_freq'      : REBALANCE_FREQ,
        'rebalance_frequency' : 'every_5_trading_days',   # ← string format Adam requires
        'max_weight_cap'      : MAX_WEIGHT,
    })

    # Validation metrics
    mlflow.log_metric('val_mse',  round(float(best_val), 8))
    mlflow.log_metric('val_rmse', round(float(best_val ** 0.5), 8))

    # Diagnostic metrics
    if '_diagnostic_metrics' in dir():
        mlflow.log_metrics(_diagnostic_metrics)

    # Backtest metrics + report artifacts
    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

    # Benchmark comparison metrics
    if '_comparison_metrics_cache' in dir():
        mlflow.log_metrics(_comparison_metrics_cache)
        print('Benchmark comparison metrics logged.')

    # ── Model submission bundle (Adam's required format) ──────────────────
    notebook_path = repo_root / 'MODELS' / 'Minho' / 'baseline_lstm.ipynb'

    log_model_submission(
        {'torch_model': str(checkpoint_path)},
        model_name          = RUN_NAME,
        model_family        = 'torch',
        feature_names       = ALL_FEATURES,
        target              = TARGET_COL,
        horizon             = HORIZON,
        rebalance_frequency = 'every_5_trading_days',
        preprocessing       = {'scaler': 'train_mean_std'},
        model_config        = {
            'architecture'      : 'BaselineLSTM',
            'input_dim'         : len(ALL_FEATURES),
            'hidden_dim'        : HIDDEN_SIZE,
            'num_layers'        : NUM_LAYERS,
            'portfolio_builder' : 'weights_from_predictions_rank_long_only',
            'required_functions': ['build_model_features', 'predict_from_prices'],
        },
        source_files = [str(notebook_path)],
    )
    print('Model submission bundle logged to MLflow.')

print(f'MLflow run "{RUN_NAME}" logged successfully.')
print('Artifacts visible in MLflow UI under: Artifacts → model_submission/')

Tracking URI: https://adams-macbook-pro.tail5ddc35.ts.net
Benchmark comparison metrics logged.
Model submission bundle logged to MLflow.
🏃 View run Minho_Baseline_LSTM at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/5/runs/5724af17ff454466b27fe22f89fdd2a4
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/5
MLflow run "Minho_Baseline_LSTM" logged successfully.
Artifacts visible in MLflow UI under: Artifacts → model_submission/


In [26]:
# ── Reproducibility verification ──────────────────────────────────
import torch
import numpy as np
import pandas as pd

def load_model_from_artifacts(checkpoint_dir: Path) -> tuple:
    """
    Reconstruct the trained model + preprocessing from the bundled checkpoint.

    Parameters
    ----------
    checkpoint_dir : Path containing lstm_checkpoint.pt

    Returns
    -------
    model          : BaselineLSTM in eval mode
    train_mean     : pd.Series of feature means
    train_std      : pd.Series of feature stds
    bundle         : full checkpoint dict
    """
    checkpoint_path = checkpoint_dir / 'lstm_checkpoint.pt'
    bundle = torch.load(checkpoint_path, map_location='cpu')

    cfg = bundle['model_config']
    reconstructed_model = BaselineLSTM(
        input_size  = cfg['input_dim'],
        hidden_size = cfg['hidden_dim'],
        num_layers  = cfg['num_layers'],
        dropout     = cfg['dropout'],
    )
    reconstructed_model.load_state_dict(bundle['state_dict'])
    reconstructed_model.eval()

    train_mean_loaded = pd.Series(bundle['preprocessing']['means'])
    train_std_loaded  = pd.Series(bundle['preprocessing']['stds'])

    return reconstructed_model, train_mean_loaded, train_std_loaded, bundle


# ── Run the verification ───────────────────────────────────────────────────────
print('Loading model from bundled checkpoint (simulating fresh session) ...')
reconstructed_model, train_mean_loaded, train_std_loaded, loaded_bundle = \
    load_model_from_artifacts(checkpoint_dir)

print(f'  Architecture  : {loaded_bundle["model_config"]}')
print(f'  Feature count : {len(loaded_bundle["feature_names"])}')
print(f'  Horizon       : {loaded_bundle["horizon"]}')
print(f'  Rebal freq    : {loaded_bundle["rebalance_frequency"]}')
print(f'  Source        : {loaded_bundle["source_notebook"]}')

# ── Reproduce predictions ──────────────────────────────────────────────────────
print('\nReproducing predictions from reconstructed model ...')
reproduced_predictions = predict_from_prices(
    model         = reconstructed_model,
    prices        = prices,
    train_mean    = train_mean_loaded,
    train_std     = train_std_loaded,
    feature_names = loaded_bundle['feature_names'],
    seq_len       = loaded_bundle['data']['seq_len'],
    horizon       = loaded_bundle['horizon'],
    tickers       = TRADABLE_TICKERS,
)

reproduced_predictions = reproduced_predictions[
    reproduced_predictions['date'].isin(test_dates)
].reset_index(drop=True)

rebal_dates_repro = set(pd.DatetimeIndex(
    sorted(reproduced_predictions['date'].unique())
)[::REBALANCE_FREQ])
reproduced_predictions = reproduced_predictions[
    reproduced_predictions['date'].isin(rebal_dates_repro)
].reset_index(drop=True)

# ── Verify match ───────────────────────────────────────────────────────────────
print('\nVerifying reproduced predictions match originals ...')
orig  = predictions.set_index(['date', 'ticker']).sort_index()
repro = reproduced_predictions.set_index(['date', 'ticker']).sort_index()

assert orig.shape == repro.shape, \
    f'Shape mismatch: original {orig.shape} vs reproduced {repro.shape}'

max_diff = float((orig['expected_return'] - repro['expected_return']).abs().max())
assert max_diff < 1e-5, \
    f'Predictions differ by up to {max_diff:.2e} — checkpoint may be corrupted'

print(f'  Rows compared     : {len(orig):,}')
print(f'  Max absolute diff : {max_diff:.2e}  (threshold: 1e-5)')
print()
print('Reproducibility check PASSED.')
print('Adam can reconstruct identical predictions from MLflow artifacts alone.')

Loading model from bundled checkpoint (simulating fresh session) ...
  Architecture  : {'architecture': 'BaselineLSTM', 'input_dim': 17, 'hidden_dim': 64, 'num_layers': 2, 'dropout': 0.2}
  Feature count : 17
  Horizon       : 5
  Rebal freq    : every_5_trading_days
  Source        : MODELS/Minho/baseline_lstm.ipynb

Reproducing predictions from reconstructed model ...

Verifying reproduced predictions match originals ...
  Rows compared     : 9,799
  Max absolute diff : 4.47e-08  (threshold: 1e-5)

Reproducibility check PASSED.
Adam can reconstruct identical predictions from MLflow artifacts alone.


In [27]:
print(f'orig shape: {orig.shape}, repro shape: {repro.shape}')
print('(assertions skipped for now)')

orig shape: (9799, 2), repro shape: (9799, 2)
(assertions skipped for now)


## 14. Final Checks

In [29]:
from IPython.display import display

# Sanity assertions
assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)
assert 'price_accel' in frame.columns

print('All checks passed. Notebook ran end to end successfully.')
print()
print('Key metrics:')
for k in ['annual_return', 'sharpe', 'max_drawdown', 'average_turnover']:
    print(f'  {k:<20s}: {result.metrics[k]:.4f}')

print()
display(result.nav.tail(3).to_frame('nav'))

All checks passed. Notebook ran end to end successfully.

Key metrics:
  annual_return       : 0.1848
  sharpe              : 1.0288
  max_drawdown        : -0.2087
  average_turnover    : 0.3255



,nav
2025-12-29,197.831531
2025-12-30,197.792962
2025-12-31,196.551321
